# Historical estimation: Glivenko-Cantelli theorem - coding <span style="color:#f55602; font-weight:bold">[20 points]</span>

<b>Reading Material</b>: Before addressing this exercise, we recommend that you study
<a style="color:#3c9591" href="https://www.arpm.co/lab/redirect.php?permalink=historical-estimation">Historical estimation</a>.

<b>Code Reference</b>: Become familiar with script s_glivenko_cantelli which illustrates the Glivenko-Cantelli theorem.

Consider an i.id. process $\varepsilon _{t}$, which is uniformly distributed
on the unit interval $\varepsilon _{t}\sim \mathit{Unif}\left( [0,1]\right)$.

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;">Prepare environment</h1>

In [1]:
from numpy import linspace, histogram, max, ones
from scipy.stats import uniform
from seaborn.distributions import ECDF  # private seaborn class (lives in seaborn._statistics);
                                        # PyCharm may underline it, but it runs - kept because it is ARPM's scaffold
from seaborn import barplot, lineplot
from matplotlib.pyplot import show

<h1 style="font-weight:bold; font-size:14px; line-height:1.5em;"> <span style="color:#f56502; font-weight:bold">[16 points]</span> Task 1 </h1>

Generate a set of $\bar{t}=10$ observations $\{\epsilon _{t}\}_{t=1}^{\bar{t}}$.
Compare the historical pdf $\hat{f}_{\varepsilon }^{\mathit{Hist}}$, as represented by a normalized histogram, with the true uniform pdf $f_{\varepsilon }$,
and compare the historical cdf $\hat{F}_{\varepsilon }^{\mathit{Hist}}$ with the true uniform cdf $F_{\varepsilon }$.

In [ ]:
################### inputs ###################
a = 0  # lower bound of uniform distribution
b = 1  # upper bound of uniform distribution
t_bar = 10  # number of observations
n_bins = 23  # number of bins for histogram
k_bar = 500  # number of grid points for cdf
##############################################

# uniform sample
# scipy parameterises Unif([a, b]) as loc=a, scale=b-a (NOT loc=a, scale=b)
eps = uniform.rvs(loc=a, scale=b - a, size=t_bar)

# uniform probabilities: each observation gets weight 1/t_bar
p = ones(t_bar) / t_bar

# historical
# pdf: normalized histogram (eq. 17.23 smoothed into bins).
# density=True divides each bin's probability by the bin width, so the AREA is 1
# and the bars are on the same scale as the true pdf (height 1 for Unif[0,1])
pdf_hist_eps, x_grid_pdf = histogram(eps, bins=n_bins, weights=p, density=True)
# cdf: historical cdf F(x) = sum_t p_t 1{eps_t <= x}  (eq. 17.24)
cdf_hist_eps, x_grid_cdf = ECDF('proportion')(eps, weights=p)

# true uniform
# pdf
pdf_eps = lambda x: uniform.pdf(x, loc=a, scale=b - a)
# cdf
cdf_eps = lambda x: uniform.cdf(x, loc=a, scale=b - a)

############################################## plots ##############################################
# grid of points
# CHANGED from ARPM's  x = linspace(0, max(eps), k_bar + 1)
# The original grid stops at the largest sample, so the "True" curves are cut off there.
# With only 10 draws max(eps) can be well below 1 (e.g. 0.8), so the true cdf never reaches 1
# and part of the true pdf is missing - exactly the region where the histogram has no data.
# Plotting over the full support [a, b] shows the whole truth we are comparing against.
x = linspace(a, b, k_bar + 1)
# historical pdf and true uniform pdf
# CHANGED from ARPM's  barplot(x=x_grid_pdf[1:], ...)
# x_grid_pdf holds the bin EDGES; x_grid_pdf[1:] are the right edges, and barplot centres
# each bar on the x it is given, so every bar was drawn half a bin too far right (the last one
# past b = 1). Centring each bar on its bin MIDPOINT puts it over the data it counts.
bin_mid = (x_grid_pdf[:-1] + x_grid_pdf[1:]) / 2
barplot(x=bin_mid, y=pdf_hist_eps, native_scale=True, color='lightgray', label='Historical')
lineplot(x=x, y=pdf_eps(x), label='True'); show();

# historical cdf and true uniform cdf
# CHANGED from ARPM's  lineplot(x=x_grid_cdf, y=cdf_hist_eps, label='Historical')
# lineplot joins the points with diagonal lines, which draws the historical cdf as a smooth curve.
# By definition (eq. 17.24) it is a STAIRCASE: flat between observations, jumping by p_t = 1/t_bar
# at each one. drawstyle='steps-post' draws it as the step function it really is.
lineplot(x=x_grid_cdf, y=cdf_hist_eps, drawstyle='steps-post', label='Historical')
lineplot(x=x, y=cdf_eps(x), label='True'); show();

### Cross-check (not part of the graded answer): our own historical cdf

`ECDF` is a private seaborn class, so it could move in a future release. Below is a
from-scratch version of eq. 17.24 that returns **exactly** the same `(cdf, grid)` pair,
plus the Glivenko-Cantelli distance $\sup_x |\hat F^{Hist}_\varepsilon(x) - F_\varepsilon(x)|$.

In [ ]:
import numpy as np


def ecdf(eps, weights=None):
    """Historical / HFP cdf, eq. 17.24 / 17.29.
    Same outputs as seaborn's ECDF('proportion')(eps, weights=...):
    (cdf values, grid), where grid = sorted data with -inf in front."""
    eps = np.asarray(eps)
    w = np.ones(len(eps)) if weights is None else np.asarray(weights)
    order = np.argsort(eps)
    grid = np.r_[-np.inf, eps[order]]
    cdf = np.r_[0.0, np.cumsum(w[order])] / w.sum()
    return cdf, grid


def sup_distance(eps, true_cdf):
    """sup_x |F_hist(x) - F(x)| (eq. 17.26). The largest gap sits just before
    or just at one of the staircase's jumps, so only those points are checked."""
    cdf, grid = ecdf(eps)
    F = true_cdf(grid[1:])
    return float(np.max(np.maximum(np.abs(cdf[1:] - F), np.abs(cdf[:-1] - F))))


own_cdf, own_grid = ecdf(eps, weights=p)
assert np.allclose(own_cdf, cdf_hist_eps) and np.array_equal(own_grid, x_grid_cdf)
print("Own ecdf matches seaborn's ECDF exactly")
print(f"t_bar = {t_bar}:  sup|F_hist - F| = {sup_distance(eps, cdf_eps):.3f}")

<h1 style="font-weight:bold; font-size:14px; line-height:1.5em;"> <span style="color:#f56502; font-weight:bold">[4 points]</span> Task 2 </h1>

Run the same test in <b>Task 1</b>, but setting $\bar{t}=1,000$.
Compare the outputs with the ones in <b>Task 1</b> and formalize the theoretical result behind. (max 30 words)

In [ ]:
t_bar = 1000  # number of observations (all other inputs as in Task 1)

eps = uniform.rvs(loc=a, scale=b - a, size=t_bar)
p = ones(t_bar) / t_bar

pdf_hist_eps, x_grid_pdf = histogram(eps, bins=n_bins, weights=p, density=True)
cdf_hist_eps, x_grid_cdf = ECDF('proportion')(eps, weights=p)

############################################## plots ##############################################
x = linspace(a, b, k_bar + 1)  # full support [a, b], see note in Task 1
bin_mid = (x_grid_pdf[:-1] + x_grid_pdf[1:]) / 2  # bin midpoints, see note in Task 1
barplot(x=bin_mid, y=pdf_hist_eps, native_scale=True, color='lightgray', label='Historical')
lineplot(x=x, y=pdf_eps(x), label='True'); show();

lineplot(x=x_grid_cdf, y=cdf_hist_eps, drawstyle='steps-post', label='Historical')  # staircase, see note in Task 1
lineplot(x=x, y=cdf_eps(x), label='True'); show();

print(f"t_bar = {t_bar}:  sup|F_hist - F| = {sup_distance(eps, cdf_eps):.3f}")

<b>Answer</b>: With $\bar{t}=1{,}000$ the histogram and cdf track the truth far more closely than with $\bar{t}=10$. Glivenko-Cantelli: for i.i.d. $\varepsilon_t$, $\sup_x|\hat{F}^{Hist}_{\varepsilon}(x)-F_{\varepsilon}(x)|\to 0$ almost surely as $\bar{t}\to\infty$.